# Research Library Repository Chroma Query

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.research_library_document_loader import ResearchLibraryChromaDocumentLoader
from apps.agentic.core.constants import RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = "../../.db"

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [2]:
RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('research_library',
 'research-library',
 '../../.db',
 ['.DS_Store', 'research_library', 'github'])

In [3]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [4]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['research-library']
Opened: research-library
total docs: 87


In [6]:
res = coll.get()   # no 'include' arg at all
print(f"docs:", len(res["ids"]))

docs: 87


## Verify Document Metadata

In [7]:
probe = coll.get(limit=5000)  # adjust as needed
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

87

In [9]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

authors
date
ext
filename
h2
images
path
section
section_char_offset
start_index
tags
title
topic


In [10]:
print(Counter(m.get("title") for m in metas if "title" in m and m.get("title")))
print("authors counts:", Counter(m.get("authors") for m in metas if "authors" in m and m.get("authors")))

Counter({'Analytic Mechanics': 87})
authors counts: Counter({'Troy Stribling': 87})


## Search Examples 

In [11]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)
hits = retriever.invoke("How are Lagrange multipliers defined?")

[print(h) for h in hits]

page_content='The method of Lagrange multipliers used to determithe the constraing forces acting on a system.  
Consider a system described by $n$ generalized coordinates $q_{1}, q_{2}, \ldots, q_{n}$. Furthure assume that the system has $k$ constraints acting on it. The constraints eleminate $k$ of the generalized coordinates leaving a total $n-k$ linearly independent coordinates.
The time behavior of the system is described by Hamilton's Principal  
$$
\delta I=\delta \int_{t_{s}}^{t_{f}} L d t=\int_{t_{s}}^{t_{f}} \delta L d t=0
$$  
with $k$ equations defining holonomic constraints,  
$$
f_{j}\left(q_{1}, q_{2}, \ldots, q_{n}\right)=0 \quad \forall \quad j=1,2, \ldots, k
$$  
consider the variation of the constraints  
$$
\delta f_{j}=\sum_{l=1}^{n} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$$  
For each $f_{j}$ define a scalar parameter $\lambda_{j}$ such that  
$$
\lambda_{j} \delta f_{j}=\sum_{l=1}^{n} \lambda_{j} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$

[None, None, None, None, None, None, None, None]

In [ ]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        # "filter": {"title": "The Evolution of Carnot’s Principle"},
    },
)
hits = retriever.invoke("Lord Kelvin's contribution to the second law of thermodynamics.")

[print(h) for h in hits]